# Tutorial 2: Composition

This tutorial builds on [Tutorial 1 (counter)](counter.ipynb). We define an inverted pendulum environment and a controller neural network, then **compose** them into a closed-loop system using shared wires.

You'll learn:
1. How modules get their own fresh wires by default
2. How to create shared wire pairs and pass them to `Env()` / `Module()` wrappers
3. How `Env` composes modules that share wires

## The inverted pendulum environment

An inverted pendulum balanced upright. The full nonlinear dynamics:

$$\ddot\theta = \frac{g}{\ell}\,\sin\theta + \frac{\tau}{m\ell^2}$$

The positive gravitational term makes the upright equilibrium ($\theta = 0$) **unstable** — without active control, any perturbation grows. The controller must apply torque $\tau$ to keep it balanced.

Write a standard `gym.Env`: `__init__` sets spaces and state, `reset` initializes, `step` updates. Then wrap with `Env()` to extract the symbolic module.

In [1]:
import math
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth.gym import Env
from zrth.torch import Module


class InvertedPendulumEnv(gym.Env):
    """Inverted pendulum with full nonlinear dynamics."""

    def __init__(self, g=9.81, l=1.0, m=1.0, dt=0.05):
        super().__init__()

        self.action_space = spaces.Box(low=-2.0, high=2.0, shape=(1,))
        self.observation_space = spaces.Box(low=-10.0, high=10.0, shape=(2,))

        self.g = g
        self.l = l
        self.m = m
        self.dt = dt

        # Also set state variables here so the analyzer can infer their dtype
        self.theta = 0.1
        self.theta_dot = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.theta = 0.1
        self.theta_dot = 0.0

        observation = [self.theta, self.theta_dot]
        return observation, {}

    def step(self, torque):
        accel = (self.g / self.l) * math.sin(self.theta) + torque / (self.m * self.l * self.l)
        self.theta_dot = self.theta_dot + accel * self.dt
        self.theta = self.theta + self.theta_dot * self.dt

        observation = [self.theta, self.theta_dot]
        reward = 0.0 - self.theta * self.theta - 0.1 * self.theta_dot * self.theta_dot
        terminated = self.theta > 3.14 or self.theta < -3.14
        truncated = False
        return observation, reward, terminated, truncated, {}

## Wrapping and composition

We create shared wire pairs up front so that the pendulum's observation feeds the controller's input, and the controller's output feeds the pendulum's action. Then we wrap both and compose them into a closed-loop system.

In [2]:
from zrth import Wire, Real

pendulum = InvertedPendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05)

# Shared wire pairs: pendulum observation (2D) = controller input
obs_wire = (Wire(Real([1, 2])), Wire(Real([1, 2])))
# Shared wire pairs: controller output = pendulum action
act_wire = (Wire(Real([1, 1])), Wire(Real([1, 1])))

wrapped_pendulum = Env(pendulum, observation=obs_wire, action=act_wire)
print(wrapped_pendulum)

module
  external
    w2, w3 : Float(1)
  interface
    w0, w1 : Float(2)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
  private
    w4, w5 : Float(1)
    w6, w7 : Float(1)
  atom controls w1, w5, w7, w13, w15, w17 reads w2, w4, w6
  init
    Tensor([9.8100004196167]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([0.05000000074505806]) w11; 
    Tensor([0.10000000149011612]) w18; 
    Id w5; w18
    Tensor([0]) w19; 
    Id w7; w19
    Stack w20; w18, w19
    Id w1; w20
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([9.8100004196167]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([0.05000000074505806]) w11; 
    Div w21; w8, w9
    Sin w22; w4
    Mul w23; w21, w22
    Mul w24; w10, w9
    Mul w25; w24, w9
    Div w26; w2, w25
    Add w27; w23, w26
    Mul w28; w27, w11
    Add w29; w6, w28
    Id w7; w29
    Mul w30; w29, w11
    Add w31; w4, w30
    Id w5; w31
    Stack w32; w31, w29
    Tenso

A small NN that observes $(\theta, \dot\theta)$ and outputs a torque $\tau$. Architecture: $[2] \to 16 \to [1]$. We wrap it with the same shared wires.

In [3]:
class ControllerNN(nn.Module):
    """NN controller: [theta, theta_dot] -> torque. Uses tanh for symmetric output."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, state):
        x = torch.tanh(self.fc1(state))
        return 2.0 * torch.tanh(self.fc2(x))  # scale to [-2, 2]

In [4]:
controller = ControllerNN()
wrapped_controller = Module(controller, extl=obs_wire, intf=act_wire)
print(wrapped_controller)

module
  external
    w0, w1 : Float(2)
  interface
    w2, w3 : Float(1)
  atom controls w3 awaits w1
  init
    Tensor([0.09818866848945618 0.0593741200864315 0.5417719483375549 0.2713233530521393 0.37888869643211365 ...]) w50; 
    Tensor([0.212734192609787 -0.23327791690826416 0.572093665599823 0.4854591190814972 -0.24763202667236328 ...]) w51; 
    Linear w52; w1, w50, w51
    Tanh w53; w52
    Tensor([2]) w54; 
    Tensor([-0.07126075029373169 0.22984609007835388 0.1833520531654358 0.0037513673305511475 0.10330331325531006 ...]) w55; 
    Tensor([0.03482219576835632]) w56; 
    Linear w57; w53, w55, w56
    Tanh w58; w57
    Mul w59; w54, w58
    Id w3; w59
  update
    Tensor([0.09818866848945618 0.0593741200864315 0.5417719483375549 0.2713233530521393 0.37888869643211365 ...]) w50; 
    Tensor([0.212734192609787 -0.23327791690826416 0.572093665599823 0.4854591190814972 -0.24763202667236328 ...]) w51; 
    Linear w52; w1, w50, w51
    Tanh w53; w52
    Tensor([2]) w54; 
    Tens

In [5]:
closed_loop = Env(wrapped_pendulum, wrapped_controller)
print(closed_loop)

module
  interface
    w0, w1 : Float(2)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
    w2, w3 : Float(1)
  private
    w4, w5 : Float(1)
    w6, w7 : Float(1)
  atom controls w1, w5, w7, w13, w15, w17 reads w2, w4, w6
  init
    Tensor([9.8100004196167]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([0.05000000074505806]) w11; 
    Tensor([0.10000000149011612]) w18; 
    Id w5; w18
    Tensor([0]) w19; 
    Id w7; w19
    Stack w20; w18, w19
    Id w1; w20
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([9.8100004196167]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([0.05000000074505806]) w11; 
    Div w21; w8, w9
    Sin w22; w4
    Mul w23; w21, w22
    Mul w24; w10, w9
    Mul w25; w24, w9
    Div w26; w2, w25
    Add w27; w23, w26
    Mul w28; w27, w11
    Add w29; w6, w28
    Id w7; w29
    Mul w30; w29, w11
    Add w31; w4, w30
    Id w5; w31
    Stack w32; w31, w29
    Tensor([0]) w33;

In [6]:
from zrth.visual import show
stop = show(closed_loop, names={obs_wire: "θ", act_wire: "τ"})

Module viewer running at http://127.0.0.1:57863 (ws://127.0.0.1:57864)


## Training the controller

We train the controller using **REINFORCE** (policy gradient). The wrapped objects are fully functional — `wrapped_pendulum.reset()` / `wrapped_pendulum.step()` work as gym methods, and `wrapped_controller(obs)` runs a PyTorch forward pass with autograd.

The controller outputs a mean torque $\mu$; we sample from $\mathcal{N}(\mu, \sigma^2)$ to explore. The loss is the standard REINFORCE estimator:

$$\nabla J = \mathbb{E}\left[\sum_t \nabla \log \pi(a_t | s_t) \cdot G_t\right]$$

where $G_t$ is the discounted return from step $t$.

In [7]:
def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns for each timestep."""
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

optimizer = torch.optim.Adam(wrapped_controller.parameters(), lr=0.01)
sigma = 0.3

for episode in range(500):
    wrapped_pendulum.reset()
    log_probs, rewards = [], []

    for step in range(200):
        state = torch.tensor([wrapped_pendulum.theta, wrapped_pendulum.theta_dot], dtype=torch.float32)
        mu = wrapped_controller(state.unsqueeze(0)).squeeze()
        dist = torch.distributions.Normal(mu, sigma)
        action = dist.sample().clamp(-2.0, 2.0)
        log_probs.append(dist.log_prob(action))

        _, reward, terminated, _, _ = wrapped_pendulum.step(action.item())
        rewards.append(reward)
        if terminated:
            break

    returns = torch.tensor(compute_returns(rewards), dtype=torch.float32)
    loss = -sum(lp * G for lp, G in zip(log_probs, returns))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if episode % 100 == 0:
        print(f"episode {episode:3d}  steps = {len(rewards):3d}  reward = {sum(rewards):.2f}")

episode   0  steps =  32  reward = -73.17


episode 100  steps =  22  reward = -78.30


episode 200  steps =  21  reward = -75.08


episode 300  steps =  21  reward = -70.68


episode 400  steps =  21  reward = -73.37


## Evaluating with the closed loop

`closed_loop.step(0)` drives the env from the controller's deterministic $\mu$ — no exploration noise. Training above was manual because REINFORCE samples actions from $\mathcal{N}(\mu, \sigma^2)$ and needs each sample's log-probability to compute the gradient.

In [8]:
closed_loop.reset()

print(f"{'step':>4}  {'theta':>8}  {'theta_dot':>10}  {'reward':>8}")
print("-" * 40)

with torch.no_grad():
    for step in range(20):
        _, reward, terminated, _, _ = closed_loop.step(0)
        print(f"{step:4d}  {closed_loop.theta:8.4f}  {closed_loop.theta_dot:10.4f}  {reward:.4f}")
        if terminated:
            print("terminated!")
            break

step     theta   theta_dot    reward
----------------------------------------
   0    0.1074      0.1483  -0.0137
   1    0.1224      0.3004  -0.0240
   2    0.1454      0.4600  -0.0423
   3    0.1770      0.6310  -0.0711
   4    0.2178      0.8172  -0.1142
   5    0.2690      1.0232  -0.1770
   6    0.3317      1.2535  -0.2671
   7    0.4073      1.5131  -0.3949
   8    0.4977      1.8074  -0.5744
   9    0.6048      2.1416  -0.8244
  10    0.7308      2.5205  -1.1694
  11    0.8782      2.9478  -1.6402
  12    1.0495      3.4253  -2.2747
  13    1.2470      3.9506  -3.1158
  14    1.4728      4.5157  -4.2082
  15    1.7280      5.1038  -5.5907
  16    2.0124      5.6882  -7.2853
  17    2.3240      6.2317  -9.2842
  18    2.6584      6.6895  -11.5422
  19    3.0093      7.0174  -13.9803
